# Chapter 15: Greedy and Sampling

[Read this chapter online](https://jackluu.io/book/section-5-generation/ch15-greedy-and-sampling/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch15-greedy-and-sampling.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 15: Greedy and Sampling

![You are here in the big picture](../assets/diagrams/ch15-where-we-are.png){ width="756" }
*Figure 15.1: Where we are: we are ready to generate new text using the trained model.*

Our model is trained and loaded. Now we want to use it to generate new text. But the model doesn't just output a single character; it produces a list of 65 scores (logits), one for each possible character in the vocabulary. We need a strategy to pick the winner. In this chapter you will:

- Generate text by always choosing the highest-scoring character (greedy decoding).
- Generate text by choosing characters randomly based on their probabilities (sampling).
- Compare the trade-offs between deterministic and creative text generation.

**Words to Know**
    - **Greedy Decoding**: always picking the single highest-probability token.
    - **Sampling**: choosing the next token randomly, giving higher-probability tokens a better chance to be selected.

## Theory

### Greedy Decoding

The simplest strategy is to always pick the character with the highest score. This is called greedy decoding. 

While it seems logical to always pick the "best" answer, greedy decoding has a major flaw: it often gets stuck in loops. Imagine the model learns a common phrase. It predicts the next character, and that character becomes part of the context. The context looks familiar, so it predicts the next character of the phrase, and soon it is repeating the same phrase forever. 

Greedy decoding is deterministic: if you give it the same prompt, it will always produce the exact same text. 

### Sampling

To avoid repetitive loops, we can use a strategy called sampling. Instead of automatically taking the top character, we turn the raw scores (logits) into percentages (probabilities) and draw a winner randomly. 

![Argmax taking the top peak vs sampling from the distribution](../assets/diagrams/ch15-greedy-and-sampling.png){ width="650" }
*Figure 15.2: Greedy decoding always takes the top peak; sampling draws randomly from the distribution.*

If the letter "e" has a 60% probability, it will be chosen 60% of the time. If "x" has a 1% probability, it is rarely chosen, but it still has a chance. This introduces variety and breaks repetitive loops, making the text feel more natural and creative. However, it also means the output is unpredictable and changes every time you run it.

As shown on the "where we are" map, generation loops the New Text back to the Tokens step: the model predicts one character, we add it to the prompt, and the cycle repeats.

**In Business**
    In our house-style assistant, the choice between greedy and sampling depends on the task. If the assistant is answering a factual question from a company manual, you want greedy decoding (safe, predictable, exact). If it is drafting a creative marketing email in the company voice, you want sampling (varied, creative, exploring new options).

## Code

We implement both strategies in the same generation loop. The core difference is how `next_id` is chosen.

```python
def generate_greedy(model, prompt, encode, decode, cfg, max_new_tokens=200):
    # Always picks the token with the highest predicted score
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    for _ in range(max_new_tokens):
        ctx     = ids[:, -cfg.block_size:]
        logits  = model(ctx)
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids     = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

def generate_sample(model, prompt, encode, decode, cfg, max_new_tokens=200):
    # Picks the next token randomly based on the model's probabilities
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    for _ in range(max_new_tokens):
        ctx    = ids[:, -cfg.block_size:]
        logits = model(ctx)
        probs  = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())
```

Line 7 picks the character with the single highest score for greedy decoding. Lines 17 and 18 turn the scores into probabilities and draw a winner randomly for sampling.

![Code flow: model(ctx) gives logits, argmax or multinomial picks next_id, which is appended to context](../assets/diagrams/ch15-code-flow.png){ width="738" }
*Figure 15.3: The generation loop predicts one character at a time and appends it to the context.*

Notice `ids[:, -cfg.block_size:]`. We crop the context to `block_size` tokens before passing it to the model. The model only learned to read a specific maximum length (our block size of 128) during training, so we must feed it at most that many tokens. We also take `logits[:, -1, :]` because we only care about the predictions for the very last character.

Let's see the two strategies in action with the prompt `ROMEO:\n`.

```python
$ python src/ch14_generate_greedy.py
--- GREEDY (always picks highest-score token) ---
ROMEO:
I will the come the some the stand the son,
And the so shall the see the shall the see the stand
...
--- SAMPLING (picks randomly from distribution) ---
ROMEO:
My must greather, but what it? whom, he ere.

EBRUTUS:
Morcy there bagainVain, I will ever may
To epated as great that the appral tankswain
...
--- Observations ---
Greedy tends to be more repetitive.
Sampling is more varied but can make unexpected choices.
```

```python
$ python diagrams/charts/ch15_probabilities.py
Saved chart: ch15-probabilities.png
```

![Bars comparing greedy and sampling on the same probabilities](../assets/diagrams/ch15-probabilities.png){ width="650" }
*Figure 15.4: Greedy always picks the tallest bar; sampling can pick any bar based on its height.*

**What just happened:**

- The greedy approach quickly got stuck in a repetitive loop ("the shall the shall...").
- The sampling approach produced varied, non-repetitive text, but it included some strange spellings ("greather", "bagainVain") because it occasionally picked low-probability characters.

### Shape Check

Table 15.1 details the shapes used during the character prediction step.

**Table 15.1:** Tensors used during the text generation loop.

| Tensor | Shape | What it means |
|--------|-------|---------------|
| `logits[:, -1, :]` | `[1, 65]` | 65 vocabulary scores for the final character in the sequence. |
| `next_id` | `[1, 1]` | 1 chosen character ID. |

## Try It

**Try It**
    Different starting prompts create different context for the model. Let's see how sampling handles new prompts.
    
    ```python title="src/examples/ch15_different_prompts.py (excerpt)" linenums="1" hl_lines="6 9"
    # ...
    encode_fn = lambda s: encode(s, char_to_id)
    decode_fn = lambda ids: decode(ids, id_to_char)
    
    print(f"--- Prompt: 'KING:\\n' ---")
    print(generate_sample(model, "KING:\n", encode_fn, decode_fn, cfg, 50))
    
    print(f"\n--- Prompt: 'JULIET:\\n' ---")
    print(generate_sample(model, "JULIET:\n", encode_fn, decode_fn, cfg, 50))
    ```
    
    Lines 6 and 9 generate new text starting from two completely different prompts.
    
    ```console title="Terminal"
    $ python src/examples/ch15_different_prompts.py
    --- Prompt: 'KING:\n' ---
    KING:
    My mine gore to kink on Villence,
    For my love grin

    --- Prompt: 'JULIET:\n' ---
    JULIET:
    What reason this bagainVain, I will evers,
    As do p

    ```

## Key Takeaways

- Greedy decoding always chooses the highest-scoring token, which leads to predictable but highly repetitive text.
- Sampling chooses the next token randomly based on the probability distribution, creating varied and natural text.
- The generation loop works by predicting one character, appending it to the context, and repeating the process.
- We crop the input context to the model's maximum `block_size` before each prediction.

## Check Your Understanding

1. Why does greedy decoding often get stuck repeating the same phrase?
2. What PyTorch function do we use to randomly pick a token based on its probabilities?
3. Why do we slice the logits tensor with `[:, -1, :]` in the generation loop?


## Further Reading

**Why always picking the likeliest word goes wrong.** Always taking the highest-scoring next token, which Chapter 15 calls greedy decoding, produces flat and repetitive text, and this paper shows why: human writing is not made of the most predictable word at every turn. Their alternative keeps the smallest set of tokens whose probabilities add up to a chosen share and samples from that. How you choose the next token matters as much as how well the model was trained.

<div class="refs" markdown>

Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2019). *The curious case of neural text degeneration* (arXiv:1904.09751). arXiv. https://doi.org/10.48550/arXiv.1904.09751

</div>

---

### `src/ch14_generate_greedy.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch14_generate_greedy.py"   # a cell has none, and the file uses it to find the text

"""
Generate text using greedy decoding and sampling.
This file belongs to Chapter 15.
Run: python src/ch14_generate_greedy.py
"""
import os
import sys
import torch
import torch.nn.functional as F


from src.utils.config import GPTConfig
from src.ch09_gpt_model import GPT

# Settings
CHECKPOINT_PATH = "checkpoints/model.pt"
DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")

# --- The Idea ---

def generate_greedy(model, prompt, encode, decode, cfg, max_new_tokens=200):
    # Always picks the token with the highest predicted score
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    for _ in range(max_new_tokens):
        ctx     = ids[:, -cfg.block_size:]
        logits  = model(ctx)
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids     = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

def generate_sample(model, prompt, encode, decode, cfg, max_new_tokens=200):
    # Picks the next token randomly based on the model's probabilities
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    for _ in range(max_new_tokens):
        ctx    = ids[:, -cfg.block_size:]
        logits = model(ctx)
        probs  = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 15: Greedy Decoding vs Sampling\n")

    if not os.path.exists(DATA_PATH):
        print("ERROR: shakespeare.txt not found. Run download_data.py")
        sys.exit(1)

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()

    chars = sorted(set(text))
    char_to_id  = {ch: i for i, ch in enumerate(chars)}
    id_to_char  = {i: ch for i, ch in enumerate(chars)}

    encode = lambda s: [char_to_id[c] for c in s if c in char_to_id]
    decode = lambda ids: "".join([id_to_char[i] for i in ids])

    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location="cpu", weights_only=False
    )
    cfg = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    trained_steps = checkpoint["step"]
    val_loss      = checkpoint["val_loss"]
    print(f"\nModel loaded: trained for {trained_steps} steps, "
          f"val loss = {val_loss:.4f}")

    prompt = "ROMEO:\n"
    print(f"\nPrompt: {repr(prompt)}")
    print("Generating 200 characters each...\n")

    with torch.no_grad():
        greedy_text = generate_greedy(
            model, prompt, encode, decode, cfg, max_new_tokens=200
        )
        sample_text = generate_sample(
            model, prompt, encode, decode, cfg, max_new_tokens=200
        )

    print("--- GREEDY (always picks highest-score token) ---")
    print(greedy_text)

    print("\n--- SAMPLING (picks randomly from distribution) ---")
    print(sample_text)

    print("\n--- Observations ---")
    print("Greedy tends to be more repetitive.")
    print("Sampling is more varied but can make unexpected choices.")
    print("For better output, see Chapter 16 (temperature and top-k).")

---

### `src/examples/ch15_different_prompts.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch15_different_prompts.py"   # a cell has none, and the file uses it to find the text

"""Generate text from different starting prompts using sampling."""
import os
import sys
import torch

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

from src.ch03_tokenizer import build_vocab, encode, decode
from src.ch09_gpt_model import GPT
from src.ch14_generate_greedy import generate_sample

def main():
    torch.manual_seed(42)
    checkpoint_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "checkpoints", "model.pt"
    )
    
    if not os.path.exists(checkpoint_path):
        print("Checkpoint not found.")
        return
        
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    
    cfg = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    
    text_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "src", "data", "shakespeare.txt"
    )
    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read()
    chars, char_to_id, id_to_char = build_vocab(text)
    
    encode_fn = lambda s: encode(s, char_to_id)
    decode_fn = lambda ids: decode(ids, id_to_char)
    
    print(f"--- Prompt: 'KING:\\n' ---")
    print(generate_sample(model, "KING:\n", encode_fn, decode_fn, cfg, 50))
    
    print(f"\n--- Prompt: 'JULIET:\\n' ---")
    print(generate_sample(model, "JULIET:\n", encode_fn, decode_fn, cfg, 50))

if __name__ == "__main__":
    main()